# Predict U-DEM

This script makes and saves predictions.

In [1]:
import os
import pickle
import pandas as pd 
import udem
import torch
import numpy as np
from torch.utils.data import DataLoader
import validation_tools as vtools

In [2]:
experiment = "v1-0"
projectDir = "/Users/rfk471/Dropbox/elevation-canada"
dataDir = f"{projectDir}/data/interim/{experiment}"

# Region ids for the entire domain
region_ids_full = ["01","02","03","04","05","06","07","08","09","10","11","12","13","14","15"]

# Region ids for the regions to be used in training and validating the model 
# These regions also contain ArcticDEM data
region_ids_model = ["01","02","03","04","05","06","12","15"]

# Region ids which are only predicted on (no ArcticDEM data for these regions)
region_ids_prediction = [x for x in region_ids_full if x not in region_ids_model] + [x for x in region_ids_model if x not in region_ids_full]

Load the model and prepare the data

In [3]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)

best_params_dir = os.path.join(projectDir,f"models/best-hpo-{experiment}.pkl")

with open(best_params_dir, "rb") as input_file:
    best_params = pickle.load(input_file)

print(f"Best parameters:\n{best_params}")

Best parameters:
{'batchnorm': True, 'dropout_on': False, 'lr': 0.0008982566968864156, 'config_idx': 9, 'epochs': 186, 'batch_size': 16, 'base_filters': 16}


In [4]:

dropout_rate = best_params["dropout_rate"] if best_params["dropout_on"] else 0.0

model = udem.UNet(
        in_channels=2,
        out_channels=1,
        int_filters=best_params["base_filters"],
        batchnorm=best_params["batchnorm"],
        dropout=dropout_rate
    ).to(device)


model.load_state_dict(torch.load(f"{projectDir}/models/unet_{experiment}.pth", map_location=device, weights_only=True))

model.eval()

UNet(
  (e1): EncoderBlock(
    (conv): ConvBlock(
      (conv): Sequential(
        (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.01, inplace=True)
        (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): LeakyReLU(negative_slope=0.01, inplace=True)
      )
      (dropout): Identity()
    )
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (e2): EncoderBlock(
    (conv): ConvBlock(
      (conv): Sequential(
        (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.01, inplace=True)
        (3): Conv2d(32, 32, ke

In [5]:
n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {n_total:,}")
print(f"Trainable parameters: {n_trainable:,}")

Total parameters:     2,205,749
Trainable parameters: 2,205,749


First, I do the predictions on the X_all dataset to get the final U-DEM product (for the regions where ArcticDEM data also exists)

In [ ]:

X_all = pd.read_pickle(f'{dataDir}/X_all.pkl')

prediction_dataset = udem.PredictionDataset(X_all)
prediction_loader = DataLoader(prediction_dataset, batch_size=best_params["batch_size"], shuffle=False)

all_preds = []

with torch.no_grad():
    for X in prediction_loader:
        X = X.to(device)

        preds = model(X)              
        preds = preds.cpu().numpy()   

        all_preds.append(preds)
all_preds = np.concatenate(all_preds, axis=0)
X_all['pred'] = list(all_preds[:,0,:,:])

Save the predicted results as netCDF files. All overlapping tiles are averaged.

In [ ]:

for region_id in region_ids_model:
    vtools.predictions_to_netcdf(X_all[X_all["region"] == region_id].copy(),projectDir,region_id,experiment,"all",0,"distance")



Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-01_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-02_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-03_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-04_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-05_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-06_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-12_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-15_v1-0_all.nc


Finally predict and save regions that contain no ArcticDEM data in them

In [ ]:

if os.path.exists(f'{dataDir}/X_all_pred.pkl'):

    X_all = pd.read_pickle(f'{dataDir}/X_all_pred.pkl')

    prediction_dataset = udem.PredictionDataset(X_all)
    prediction_loader = DataLoader(prediction_dataset, batch_size=best_params["batch_size"], shuffle=False)

    all_preds = []

    with torch.no_grad():
        for X in prediction_loader:
            X = X.to(device)

            preds = model(X)             
            preds = preds.cpu().numpy() 

            all_preds.append(preds)
    all_preds = np.concatenate(all_preds, axis=0)
    X_all['pred'] = list(all_preds[:,0,:,:])


    for region_id in region_ids_prediction:
        vtools.predictions_to_netcdf(X_all[X_all["region"] == region_id].copy(),projectDir,region_id,experiment,"all",0,"distance")

Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-07_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-08_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-09_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-10_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-11_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-13_v1-0_all.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-14_v1-0_all.nc


Create netCDF files of the training data (potentially used for plotting)

In [ ]:

X_train = pd.read_pickle(f'{dataDir}/X_train.pkl')
y_train = pd.read_pickle(f'{dataDir}/y_train.pkl')
X_val = pd.read_pickle(f'{dataDir}/X_val.pkl')
y_val = pd.read_pickle(f'{dataDir}/y_val.pkl')
X_train["pred"] = y_train["adem"]
X_val["pred"] = y_val["adem"]
X_all = pd.concat([X_train,X_val])

for region_id in region_ids_model:
    vtools.predictions_to_netcdf(X_all[X_all["region"] == region_id].copy(),projectDir,region_id,experiment,"train",0,"distance")

Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-01_v1-0_train.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-02_v1-0_train.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-03_v1-0_train.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-04_v1-0_train.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-05_v1-0_train.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-06_v1-0_train.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-12_v1-0_train.nc
Saved: /Users/rfk471/Dropbox/elevation-canada/data/final/udem_region-15_v1-0_train.nc
